# AIMA Skin Lesion Segmentation - Kaggle controlled rerun

This notebook runs the repaired, modular pipeline on Kaggle without embedding the implementation in notebook cells.

It performs these stages:

1. Extract and install the repaired repository.
2. Inspect the Kaggle runtime and attached dataset.
3. Run the synthetic correctness test suite.
4. Validate image-mask pairing and create the split manifest.
5. Optionally train the Attention U-Net.
6. Inspect validation metrics, qualitative outputs, and reproducibility artifacts.
7. Optionally generate a sample-submission-validated `submission.csv`.

The historical scores are not evidence for this repaired pipeline. A new score should be reported only after this controlled rerun finishes and its artifacts are saved.

**Safety gates:** training and submission are disabled by default. Edit the first code cell, run through `prepare`, inspect the manifests, then enable training. Submission remains blocked until the official RLE order is confirmed.

## 1. User settings

Usually, only this cell needs editing. The default dataset paths come from the historical Kaggle notebook and must still be checked against the attached input tree.

Recommended first pass:

- Keep `RUN_TRAINING = False`.
- Keep `RUN_SUBMISSION = False`.
- Run through the preparation and manifest-inspection cells.
- Only then enable training.


In [ ]:
from pathlib import Path

# Repository archive attached as a private Kaggle Dataset.
REPO_ARCHIVE_NAME = "aima-skin-lesion-segmentation-repaired.zip"

# Paths recorded in the historical Kaggle notebook. Verify them below.
DATASET_ROOT = Path("/kaggle/input/warm-up-program-ai-vietnam-skin-segmentation")
TRAIN_IMAGE_DIR = DATASET_ROOT / "Train/Train/Image"
MASK_DIR = DATASET_ROOT / "Train/Train/Mask"
TEST_IMAGE_DIR = DATASET_ROOT / "Test/Test/Image"

# Set an exact path when known. If None, the notebook lists CSV candidates.
SAMPLE_SUBMISSION_PATH = None
GROUP_MAPPING_PATH = None  # Optional JSON mapping or CSV with sample_id,group_id.

# "kaggle": keep Kaggle's TensorFlow and install the repaired project plus pinned support packages.
# "strict": install every version from requirements-dev.txt. This is more reproducible but may require a session restart.
# "none": install nothing; useful only when the environment is already prepared.
DEPENDENCY_MODE = "kaggle"
RUN_REPOSITORY_TESTS = True
RUN_TRAINING = False
RUN_SUBMISSION = False

# Controlled-run configuration.
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 256
BATCH_SIZE = 8
EPOCHS = 50
BASE_FILTERS = 32
EARLY_STOPPING_PATIENCE = 8
VALIDATION_FRACTION = 0.20
SEED = 42
LEARNING_RATE = 1e-3
L2_COEFFICIENT = 1e-4
MIXED_PRECISION = True
N_TTA = 8

THRESHOLD_GRID = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
MIN_COMPONENT_SIZES = [0, 16, 32, 64]
MORPHOLOGY_KERNELS = [0, 3]
MASK_SUFFIX = "_segmentation"

# These are derived from the official sample submission when available.
SUBMISSION_ID_COLUMN = None
SUBMISSION_MASK_COLUMN = None
SUBMISSION_ID_SUFFIX = None

# Do not set this True until the competition specification or official starter code confirms the order.
RLE_ORDER = "C"
RLE_ORDER_CONFIRMED = False

WORK_ROOT = Path("/kaggle/working")
OUTPUT_DIR = WORK_ROOT / "artifacts" / "controlled_rerun"
CONFIG_PATH = WORK_ROOT / "kaggle_controlled_rerun.json"

print("Training enabled:", RUN_TRAINING)
print("Submission enabled:", RUN_SUBMISSION)
print("Output directory:", OUTPUT_DIR)

## 2. Inspect the Kaggle runtime and attached inputs

In [ ]:
import os
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())
print("Kaggle input exists:", Path("/kaggle/input").exists())
print("Kaggle working exists:", WORK_ROOT.exists())

subprocess.run(["nvidia-smi"], check=False)

print("\nTop-level Kaggle inputs:")
for path in sorted(Path("/kaggle/input").glob("*")):
    print(" -", path)

## 3. Locate and extract the repaired repository

The archive can be inside any attached Kaggle Dataset. The cell requires exactly one matching archive to avoid extracting the wrong project.

In [ ]:
import shutil

archive_matches = sorted(Path("/kaggle/input").rglob(REPO_ARCHIVE_NAME))
if len(archive_matches) != 1:
    raise RuntimeError(
        f"Expected exactly one {REPO_ARCHIVE_NAME!r} under /kaggle/input, found {archive_matches}"
    )

repo_archive = archive_matches[0]
extract_root = WORK_ROOT / "repaired_repository"
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
shutil.unpack_archive(str(repo_archive), str(extract_root))

project_matches = sorted(path.parent for path in extract_root.rglob("pyproject.toml"))
if len(project_matches) != 1:
    raise RuntimeError(f"Expected one extracted project, found {project_matches}")
REPO_ROOT = project_matches[0]

os.chdir(REPO_ROOT)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

print("Repository archive:", repo_archive)
print("Repository root:", REPO_ROOT)
print("Git metadata present:", (REPO_ROOT / ".git").is_dir())

## 4. Install dependencies and the project

`DEPENDENCY_MODE = "kaggle"` is the practical default. It keeps Kaggle's GPU-enabled TensorFlow build, installs the pinned augmentation/test packages, and installs this repository without resolving all pinned dependencies again.

Use `"strict"` only when the Kaggle Python version supports the pinned requirements. If pip replaces TensorFlow, Keras, or NumPy, restart the notebook session and rerun from the top before training.

In [ ]:
def run_command(command, *, cwd=REPO_ROOT):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    completed = subprocess.run(command, cwd=cwd, text=True)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {' '.join(command)}")
    return completed

if DEPENDENCY_MODE == "strict":
    run_command([
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
        "-r", REPO_ROOT / "requirements-dev.txt",
    ])
elif DEPENDENCY_MODE == "kaggle":
    run_command([
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
        "albumentations==1.4.24",
        "opencv-python-headless==4.10.0.84",
        "pytest==8.3.3",
    ])
elif DEPENDENCY_MODE != "none":
    raise ValueError("DEPENDENCY_MODE must be 'kaggle', 'strict', or 'none'")

run_command([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", REPO_ROOT, "--no-deps",
])

## 5. Record actual package and hardware versions

The run artifacts also record the environment automatically. This cell provides an early check before tests or training.

In [ ]:
import importlib.metadata

packages = [
    "tensorflow",
    "keras",
    "numpy",
    "opencv-python-headless",
    "albumentations",
    "pandas",
    "matplotlib",
    "pytest",
]
for package in packages:
    try:
        print(f"{package}: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

import tensorflow as tf
print("TensorFlow devices:", tf.config.list_physical_devices())
print("GPU devices:", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    print("WARNING: No GPU is visible. Do not start the full training run.")

## 6. Run correctness checks

The ordinary test suite uses synthetic data and does not need the private dataset. With TensorFlow and Albumentations installed, the previously dependency-gated tests should run rather than skip.

The exact warning-strict command is `pytest -q -W error`.


In [ ]:
if RUN_REPOSITORY_TESTS:
    run_command([sys.executable, "-m", "pytest", "-q", "-W", "error"])
    run_command([sys.executable, "scripts/verify_repository.py"])
    run_command([sys.executable, "scripts/synthetic_smoke.py"])
else:
    print("Repository tests skipped by configuration.")

## 7. Inspect the dataset layout

This cell does not guess patient or lesion IDs. It only checks the configured directories, counts supported files, and lists CSV candidates. If the default paths are wrong, edit the first code cell and rerun from there.

In [ ]:
from skin_lesion_segmentation.data import SUPPORTED_IMAGE_EXTENSIONS


def supported_files(directory):
    directory = Path(directory)
    if not directory.is_dir():
        return []
    return sorted(
        path for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
    )

for label, directory in [
    ("Training images", TRAIN_IMAGE_DIR),
    ("Training masks", MASK_DIR),
    ("Test images", TEST_IMAGE_DIR),
]:
    files = supported_files(directory)
    print(f"{label}: {directory}")
    print(f"  exists={directory.is_dir()}, supported_files={len(files)}")
    if files:
        print("  first:", files[0].name)
        print("  last: ", files[-1].name)

missing_required = [
    str(path) for path in (TRAIN_IMAGE_DIR, MASK_DIR, TEST_IMAGE_DIR) if not Path(path).is_dir()
]
if missing_required:
    print("\nCandidate image directories under DATASET_ROOT:")
    if DATASET_ROOT.exists():
        for directory in sorted(path for path in DATASET_ROOT.rglob("*") if path.is_dir()):
            count = len(supported_files(directory))
            if count:
                print(f" - {directory}: {count} supported images")
    raise FileNotFoundError("Configured dataset directories do not exist: " + ", ".join(missing_required))

csv_candidates = sorted(DATASET_ROOT.rglob("*.csv")) if DATASET_ROOT.exists() else []
print("\nCSV candidates:")
for path in csv_candidates:
    print(" -", path)

## 8. Inspect the official sample submission and derive its ID contract

The notebook may infer a constant suffix such as `_segmentation` only when every official sample-submission ID proves the same mapping from the test image stems. A non-uniform mapping fails instead of being guessed.

In [ ]:
import pandas as pd
from skin_lesion_segmentation.inference import discover_test_images
from skin_lesion_segmentation.submission import infer_submission_id_suffix

if SAMPLE_SUBMISSION_PATH is None:
    likely_samples = [
        path for path in csv_candidates
        if "sample" in path.name.casefold() and "submission" in path.name.casefold()
    ]
    if len(likely_samples) == 1:
        SAMPLE_SUBMISSION_PATH = likely_samples[0]
    elif len(csv_candidates) == 1:
        SAMPLE_SUBMISSION_PATH = csv_candidates[0]

if SAMPLE_SUBMISSION_PATH is None:
    print("No unique sample submission was found. Training can continue, but submission generation will remain disabled.")
    sample_submission = None
    if SUBMISSION_ID_SUFFIX is None:
        SUBMISSION_ID_SUFFIX = ""
else:
    SAMPLE_SUBMISSION_PATH = Path(SAMPLE_SUBMISSION_PATH)
    if not SAMPLE_SUBMISSION_PATH.is_file():
        raise FileNotFoundError(SAMPLE_SUBMISSION_PATH)
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH, keep_default_na=False)
    print("Sample submission:", SAMPLE_SUBMISSION_PATH)
    print("Columns:", sample_submission.columns.tolist())
    print("Rows:", len(sample_submission))
    display(sample_submission.head())

    if len(sample_submission.columns) != 2:
        raise ValueError("Expected the official sample submission to contain exactly two columns")
    if SUBMISSION_ID_COLUMN is None:
        SUBMISSION_ID_COLUMN = str(sample_submission.columns[0])
    if SUBMISSION_MASK_COLUMN is None:
        SUBMISSION_MASK_COLUMN = str(sample_submission.columns[1])
    if sample_submission.columns.tolist() != [SUBMISSION_ID_COLUMN, SUBMISSION_MASK_COLUMN]:
        raise ValueError("Configured submission columns do not match the official sample submission order")

    test_paths = discover_test_images(TEST_IMAGE_DIR)
    inferred_suffix = infer_submission_id_suffix(
        test_paths,
        sample_submission,
        id_column=SUBMISSION_ID_COLUMN,
    )
    if SUBMISSION_ID_SUFFIX is None:
        SUBMISSION_ID_SUFFIX = inferred_suffix
    elif SUBMISSION_ID_SUFFIX != inferred_suffix:
        raise ValueError(
            f"Configured submission suffix {SUBMISSION_ID_SUFFIX!r} does not match template-proven suffix {inferred_suffix!r}"
        )
    print("Submission ID column:", SUBMISSION_ID_COLUMN)
    print("Submission mask column:", SUBMISSION_MASK_COLUMN)
    print("Template-proven ID suffix:", repr(SUBMISSION_ID_SUFFIX))

## 9. Write and validate the effective configuration

Without reliable external group metadata, this remains an image-level split. Patient or lesion independence cannot be guaranteed.

In [ ]:
import json
from skin_lesion_segmentation.config import ExperimentConfig

config_data = {
    "image_dir": str(TRAIN_IMAGE_DIR),
    "mask_dir": str(MASK_DIR),
    "test_image_dir": str(TEST_IMAGE_DIR),
    "sample_submission_path": str(SAMPLE_SUBMISSION_PATH) if SAMPLE_SUBMISSION_PATH is not None else None,
    "group_mapping_path": str(GROUP_MAPPING_PATH) if GROUP_MAPPING_PATH is not None else None,
    "output_dir": str(OUTPUT_DIR),
    "mask_suffix": MASK_SUFFIX,
    "image_height": IMAGE_HEIGHT,
    "image_width": IMAGE_WIDTH,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "base_filters": BASE_FILTERS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "validation_fraction": VALIDATION_FRACTION,
    "seed": SEED,
    "learning_rate": LEARNING_RATE,
    "l2_coefficient": L2_COEFFICIENT,
    "mixed_precision": MIXED_PRECISION,
    "rle_order": RLE_ORDER,
    "rle_order_confirmed": RLE_ORDER_CONFIRMED,
    "submission_id_column": SUBMISSION_ID_COLUMN or "id",
    "submission_mask_column": SUBMISSION_MASK_COLUMN or "segmentation",
    "submission_id_suffix": SUBMISSION_ID_SUFFIX or "",
    "n_tta": N_TTA,
    "threshold_grid": THRESHOLD_GRID,
    "min_component_sizes": MIN_COMPONENT_SIZES,
    "morphology_kernels": MORPHOLOGY_KERNELS,
}

config = ExperimentConfig(**config_data)
config.validate()
config.save(CONFIG_PATH)
print(CONFIG_PATH.read_text())

## 10. Prepare pairing and split manifests

This is the mandatory checkpoint before training. It validates canonical image-mask pairing, exact decoded-image duplicates, and group leakage when explicit metadata is supplied.

CLI form: `python -m skin_lesion_segmentation.cli prepare --config ...`.


In [ ]:
run_command([
    sys.executable, "-m", "skin_lesion_segmentation.cli",
    "prepare", "--config", CONFIG_PATH,
])

pair_manifest_path = OUTPUT_DIR / "pair_manifest.csv"
split_manifest_csv_path = OUTPUT_DIR / "split_manifest.csv"
split_manifest_json_path = OUTPUT_DIR / "split_manifest.json"

pair_manifest = pd.read_csv(pair_manifest_path)
split_manifest = pd.read_csv(split_manifest_csv_path, keep_default_na=False)

print("Paired samples:", len(pair_manifest))
print("Split counts:")
print(split_manifest["split"].value_counts())
print("Group metadata present:", bool((split_manifest["group_id"].astype(str).str.len() > 0).any()))
print("Exact duplicate clusters:", int((split_manifest.groupby("image_sha256").size() > 1).sum()))

display(pair_manifest.head())
display(split_manifest.head())

## 11. Pre-training smoke check

This creates the configured model and performs one CPU/GPU forward pass plus loss calculation. It does not train the model.

In [ ]:
import numpy as np
from skin_lesion_segmentation.losses import combined_segmentation_loss
from skin_lesion_segmentation.model import build_attention_unet

smoke_model = build_attention_unet(
    (IMAGE_HEIGHT, IMAGE_WIDTH, 3),
    base_filters=BASE_FILTERS,
    l2_coefficient=L2_COEFFICIENT,
    mixed_precision=MIXED_PRECISION,
)
smoke_images = np.zeros((1, IMAGE_HEIGHT, IMAGE_WIDTH, 3), dtype=np.float32)
smoke_masks = np.zeros((1, IMAGE_HEIGHT, IMAGE_WIDTH, 1), dtype=np.float32)
smoke_predictions = smoke_model(smoke_images, training=False)
smoke_loss = combined_segmentation_loss(smoke_masks, smoke_predictions)

print("Output shape:", tuple(smoke_predictions.shape))
print("Output dtype:", smoke_predictions.dtype)
print("Loss dtype:", smoke_loss.dtype)
print("Loss finite:", bool(np.isfinite(float(smoke_loss.numpy()))))

del smoke_model, smoke_images, smoke_masks, smoke_predictions

## 12. Controlled training

Training is intentionally gated. Review the manifest output first. Then change `RUN_TRAINING = True` in the first settings cell and rerun this cell.

Checkpoint selection uses minimum validation loss. The selected `.keras` checkpoint is reloaded before source-of-truth offline validation.

CLI form: `python -m skin_lesion_segmentation.cli train --config ...`.


In [ ]:
if RUN_TRAINING:
    if not tf.config.list_physical_devices("GPU"):
        raise RuntimeError("RUN_TRAINING is True but no GPU is visible")
    run_command([
        sys.executable, "-m", "skin_lesion_segmentation.cli",
        "train", "--config", CONFIG_PATH,
    ])
else:
    print("Training not started. Set RUN_TRAINING = True after reviewing the manifests.")

## 13. Inspect training history and reproducibility artifacts

In [ ]:
import matplotlib.pyplot as plt

print("Artifact directory:", OUTPUT_DIR)
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(f" - {path.name}: {path.stat().st_size} bytes")

history_path = OUTPUT_DIR / "training_history.csv"
if history_path.exists():
    history = pd.read_csv(history_path)
    display(history.tail())

    if {"loss", "val_loss"}.issubset(history.columns):
        plt.figure(figsize=(8, 4))
        plt.plot(history["epoch"], history["loss"], label="training loss")
        plt.plot(history["epoch"], history["val_loss"], label="validation loss")
        plt.xlabel("Epoch")
        plt.ylabel("Keras loss")
        plt.legend()
        plt.grid(alpha=0.2)
        plt.show()
else:
    print("No training history exists yet.")

## 14. Inspect offline validation metrics and qualitative cases

The final metrics are thresholded per image and macro-averaged. Raw threshold-0.5 results and validation-selected post-processing results are reported separately.

In [ ]:
from IPython.display import Image, display as display_image

metrics_path = OUTPUT_DIR / "final_validation_metrics.json"
postprocessing_path = OUTPUT_DIR / "chosen_postprocessing.json"
qualitative_path = OUTPUT_DIR / "qualitative_validation.png"
predictions_path = OUTPUT_DIR / "validation_predictions.npz"

if metrics_path.exists():
    final_metrics = json.loads(metrics_path.read_text())
    print(json.dumps(final_metrics, indent=2))
else:
    print("No final validation metrics exist yet.")

if postprocessing_path.exists():
    print("\nChosen validation-only post-processing:")
    print(postprocessing_path.read_text())

if qualitative_path.exists():
    display_image(Image(filename=str(qualitative_path)))
else:
    print("No qualitative validation grid exists yet.")

if predictions_path.exists():
    run_command([
        sys.executable, "-m", "skin_lesion_segmentation.cli",
        "evaluate-predictions", "--predictions", predictions_path,
        "--threshold", "0.5",
    ])

## 15. Verify checkpoint reload metadata

In [ ]:
checkpoint_path = OUTPUT_DIR / "best_model.keras"
checkpoint_record_path = OUTPUT_DIR / "checkpoint.json"

if checkpoint_path.exists() and checkpoint_record_path.exists():
    checkpoint_record = json.loads(checkpoint_record_path.read_text())
    print(json.dumps(checkpoint_record, indent=2))
    run_command([sys.executable, "scripts/checkpoint_checksum.py", checkpoint_path])
else:
    print("No selected checkpoint exists yet.")

## 16. Confirm the official RLE order before submission

The historical notebook used NumPy's default C-order flattening, but that alone is not proof of the official evaluator contract. Confirm the order from the official competition specification, official starter code, evaluator, or a known asymmetric example.

After confirmation:

1. Set `RLE_ORDER` to `"C"` or `"F"` in the first settings cell.
2. Set `RLE_ORDER_CONFIRMED = True`.
3. Rerun the configuration-writing cell.
4. Set `RUN_SUBMISSION = True`.
5. Run the submission cell below.


## 17. Generate and validate `submission.csv`

Generation is blocked unless all of these exist:

- Official sample submission.
- Selected checkpoint.
- Validation-selected post-processing parameters.
- Template-proven test-ID mapping.
- Explicitly confirmed RLE order.


CLI form: `python -m skin_lesion_segmentation.cli submit --config ...`.


In [ ]:
if RUN_SUBMISSION:
    if SAMPLE_SUBMISSION_PATH is None:
        raise RuntimeError("SAMPLE_SUBMISSION_PATH is required")
    if not RLE_ORDER_CONFIRMED:
        raise RuntimeError("RLE order has not been officially confirmed")
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    if not postprocessing_path.is_file():
        raise FileNotFoundError(postprocessing_path)

    run_command([
        sys.executable, "-m", "skin_lesion_segmentation.cli",
        "submit", "--config", CONFIG_PATH,
        "--checkpoint", checkpoint_path,
        "--postprocessing", postprocessing_path,
    ])

    submission_path = OUTPUT_DIR / "submission.csv"
    submission_summary_path = OUTPUT_DIR / "submission_validation_summary.json"
    submission = pd.read_csv(submission_path, keep_default_na=False)
    display(submission.head())
    print(json.dumps(json.loads(submission_summary_path.read_text()), indent=2))
else:
    print("Submission not generated. Confirm the official RLE order, then set RUN_SUBMISSION = True.")

## 18. Package the controlled-run evidence

Kaggle preserves `/kaggle/working` when you save a notebook version. This cell also creates one downloadable archive containing the actual generated artifacts. It does not create placeholder results.

In [ ]:
artifact_archive = WORK_ROOT / "aima_skin_lesion_controlled_rerun_artifacts.zip"
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    if artifact_archive.exists():
        artifact_archive.unlink()
    shutil.make_archive(
        str(artifact_archive.with_suffix("")),
        "zip",
        root_dir=OUTPUT_DIR,
    )
    print("Artifact archive:", artifact_archive)
    print("Size:", artifact_archive.stat().st_size, "bytes")
else:
    print("No run artifacts are available to package.")

## 19. What to save after the run

Use Kaggle's **Save Version** action after training finishes. Preserve at least:

- The completed notebook output.
- `best_model.keras` and `checkpoint.json`.
- `effective_config.json`, `environment.json`, and `random_seed.json`.
- Pair and split manifests.
- `training_history.csv`.
- `validation_predictions.npz`.
- `final_validation_metrics.json` and `chosen_postprocessing.json`.
- `qualitative_validation.png`.
- `submission.csv` and `submission_validation_summary.json`, only if submission was generated.

Do not rewrite the README with a repaired-pipeline score until these artifacts have been reviewed. Do not use public or private leaderboard feedback to retune thresholding or post-processing.